<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/Taller_Control_1.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg"></a>

# Taller de Control 1 — S1 a S6
**Big Data → arquitectura → MongoDB → Cassandra → Neo4j**  
Individual · 180 min · 100 puntos · **Git/GitHub opcional**

## Entrega
1. cuaderno `.ipynb` ejecutado;
2. `TC1_<codigo>_resultados.json`;
3. `TC1_<codigo>_reporte.md`.

No se requieren Atlas, Astra ni Aura para obtener 100 puntos. Trabaja solo en celdas **TU CÓDIGO / TU RESPUESTA**. Los datos son versionados y los controles son determinísticos. Las celdas **NO EDITAR** aparecen plegadas.

| Bloque | Pts |
|---|---:|
| A Fundamentos | 15 |
| B Ingesta | 15 |
| C MongoDB | 20 |
| D Cassandra + bandeja | 25 |
| E Neo4j | 20 |
| F Integridad | 5 |

In [ ]:
# NO EDITAR
from pathlib import Path
import json,re,hashlib,urllib.request
import pandas as pd

ESTUDIANTE=input("Nombre: " ).strip(); CODIGO=input("Código: " ).strip()
if not ESTUDIANTE or not CODIGO: raise ValueError("Completa nombre y código")
R={}
def ev(k,ok,p,d=""):
    R[k]={"ok":bool(ok),"puntos":p if ok else 0,"maximo":p,"detalle":str(d)}
    print(("✅" if ok else "❌"),k,R[k]["puntos"],"/",p,d)
RAW="https://raw.githubusercontent.com/jazaineam1/BigData2026/main"
secop=pd.read_csv(f"{RAW}/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",low_memory=False)
with urllib.request.urlopen(f"{RAW}/Datos/noticias_contratacion_2026.json") as x: noticias=json.loads(x.read())
with urllib.request.urlopen(f"{RAW}/Datos/entidades_en_noticias_2026.json") as x: menciones=json.loads(x.read())
s06=pd.read_csv(f"{RAW}/Datos/s06_contexto_relacional.csv",low_memory=False)
print(len(secop),len(noticias),len(menciones),len(s06))

## A. Fundamentos — 15 puntos
Escribe una letra por pregunta.

**A1.** Llegan 3 M eventos/min y proceso 1 M/min: A variedad · B velocidad/capacidad · C veracidad · D visualización.  
**A2.** Tablas+JSON+texto+imágenes: A volumen · B valor · C variedad · D veracidad.  
**A3.** Antes de herramienta: A cloud · B decisión/usuario/evidencia · C lenguaje · D clúster.  
**A4.** Documentos irregulares: A documental · B hoja · C Cassandra sin consulta · D grafo sin relaciones.  
**A5.** Query-first: A normalizar primero · B diseñar desde consulta · C PK universal · D JSON.

In [ ]:
# TU RESPUESTA
A={"A1":"","A2":"","A3":"","A4":"","A5":""}

In [ ]:
# NO EDITAR
K={"A1":"B","A2":"C","A3":"B","A4":"A","A5":"B"}
ac=sum(str(A[k]).upper()==v for k,v in K.items())
ev("A_fundamentos",ac==5,15,f"{ac}/5")

## B. Ingesta — 15 puntos
Calcula, **sin escribir manualmente el resultado**:
- `n_secop`: filas de `secop`;
- `n_noticias`: elementos de `noticias`;
- `n_largas`: noticias con `n_palabras > 800`.

In [ ]:
# TU CÓDIGO
n_secop=None
n_noticias=None
n_largas=None

In [ ]:
# NO EDITAR
ev("B1",n_secop==1000,5,n_secop)
ev("B2",n_noticias==987,5,n_noticias)
ev("B3",n_largas==189,5,n_largas)

## C. MongoDB documental — 20 puntos
Se usa `mongomock` para evaluar MQL sin depender de una cuenta externa.

1. `filtro_largas`: más de 800 palabras.
2. `filtro_bogota`: `seccion=="bogota"` y más de 500 palabras.
3. `proyeccion_titulo`: mostrar `titulo`, ocultar `_id`.
4. `pipeline`: `$match n_palabras>0` → `$group` por `seccion` con `noticias` y `promedio_palabras` → `$sort` noticias DESC, `_id` ASC → `$limit 10`.

In [ ]:
# NO EDITAR
!pip -q install mongomock
import mongomock
col=mongomock.MongoClient().db.noticias
col.insert_many(noticias)

In [ ]:
# TU CÓDIGO
filtro_largas={}
filtro_bogota={}
proyeccion_titulo={}
pipeline=[]
conteo_largas=col.count_documents(filtro_largas)
res_pipeline=list(col.aggregate(pipeline))

In [ ]:
# NO EDITAR
ref_b={"seccion":"bogota","n_palabras":{"$gt":500}}
ref_p={"_id":0,"titulo":1}
ref_pipe=[
 {"$match":{"n_palabras":{"$gt":0}}},
 {"$group":{"_id":"$seccion","noticias":{"$sum":1},"promedio_palabras":{"$avg":"$n_palabras"}}},
 {"$sort":{"noticias":-1,"_id":1}},{"$limit":10}]
def canon(z):
    o=[]
    for d in z:
        d=dict(d)
        if d.get("promedio_palabras") is not None:d["promedio_palabras"]=round(float(d["promedio_palabras"]),7)
        o.append(d)
    return o
ev("C1_MQL",conteo_largas==189,5,conteo_largas)
ev("C2_proyeccion",list(col.find(filtro_bogota,proyeccion_titulo).limit(20))==list(col.find(ref_b,ref_p).limit(20)),5)
ev("C3_aggregate",canon(res_pipeline)==canon(list(col.aggregate(ref_pipe))),10,len(res_pipeline))

## D. Bandeja S5 + Cassandra query-first — 25 puntos

### D1–D2
Crea `contexto_menciones` con columnas `entidad, noticias_entidad, nivel_menciones` (renombra `noticias`). Luego:
`paso1` = SECOP de entidades presentes → `paso2` = contratación que contiene “directa” → `paso3` = `respuestas_al_procedimiento==0` → `candidatos` = left merge con contexto.

### D3
Consulta Cassandra: **para `corte`+`departamento`, top 5 por `valor_base`**. Completa partición, clustering y orden.

In [ ]:
# TU CÓDIGO
contexto_menciones=None
entidades_en_prensa=None
paso1=None
paso2=None
paso3=None
candidatos=None

# TU RESPUESTA CASSANDRA
PARTICION=""
CLUSTER=""
ORDEN=""

In [ ]:
# NO EDITAR
ok1=(contexto_menciones is not None and list(contexto_menciones.columns)==["entidad","noticias_entidad","nivel_menciones"]
     and len(contexto_menciones)==142 and contexto_menciones["entidad"].is_unique)
ok2=(paso1 is not None and paso2 is not None and paso3 is not None and candidatos is not None
     and len(paso1)==163 and len(candidatos)==77
     and candidatos["noticias_entidad"].notna().all() and candidatos["nivel_menciones"].notna().all())
ok3=(re.sub(r"\s+","",PARTICION.lower())=="corte,departamento" and CLUSTER.lower().strip()=="valor_base" and ORDEN.upper().strip()=="DESC")
ev("D1_contexto",ok1,5,0 if contexto_menciones is None else len(contexto_menciones))
ev("D2_bandeja",ok2,10,None if candidatos is None else len(candidatos))
ev("D3_cassandra",ok3,10,f"(({PARTICION}),{CLUSTER}) {ORDEN}")

## E. Neo4j — 20 puntos

Ancla fija: **FUERZA AEROESPACIAL COLOMBIANA**, NIT `899999102`.

**E1.** Completa labels y relaciones exactos del curso.  
**E2.** Completa operador de desigualdad Cypher.  
**E3.** Con `s06`: `hist` = filas con `nit_proveedor`; `hist_ancla` = NIT ancla; por proveedor calcula `procesos_con_entidad` (`id_proceso` únicos) y `entidades_conectadas` (`nit_entidad` únicos globales); une, ordena `entidades_conectadas DESC`, `procesos_con_entidad DESC`, `nit_proveedor ASC`; guarda máximo en `maximo_h2r`.  
**E4.** Límite: A colusión · B favorecimiento · C relaciones registradas no prueban irregularidad · D prensa implica irregularidad.

In [ ]:
# TU RESPUESTA / TU CÓDIGO
LABEL_ENTIDAD=""; LABEL_PROCESO=""; LABEL_PROVEEDOR=""
REL1=""; REL2=""; OPERADOR=""
hist=None; hist_ancla=None; resultado_h2r=None; maximo_h2r=None
LIMITE=""

In [ ]:
# NO EDITAR
e1=(LABEL_ENTIDAD=="Entidad" and LABEL_PROCESO=="Proceso" and LABEL_PROVEEDOR=="Proveedor" and REL1=="PUBLICA" and REL2=="ADJUDICADO_A")
e2=OPERADOR.strip()=="<>"
e3=(hist is not None and hist_ancla is not None and resultado_h2r is not None and maximo_h2r is not None and int(maximo_h2r)==39)
e4=LIMITE.upper().strip()=="C"
ev("E1_modelo",e1,5); ev("E2_cypher",e2,5); ev("E3_h2r",e3,5,maximo_h2r); ev("E4_limite",e4,5)

## F. Generar entrega — 5 puntos
Ejecuta. El JSON registra cada control y una huella SHA-256. GitHub sigue siendo opcional.

In [ ]:
# NO EDITAR
esperados=["A_fundamentos","B1","B2","B3","C1_MQL","C2_proyeccion","C3_aggregate","D1_contexto","D2_bandeja","D3_cassandra","E1_modelo","E2_cypher","E3_h2r","E4_limite"]
ev("F_integridad",all(k in R for k in esperados),5)
pts=sum(v["puntos"] for v in R.values()); nota=round(1+4*pts/100,2)
ent={"taller":"TC1 S1-S6","version":"2026-09-17-v2","estudiante":ESTUDIANTE,"codigo":CODIGO,"puntaje":pts,"nota_5":nota,"controles":R,
     "control":{"secop":len(secop),"noticias":len(noticias),"paso1":None if paso1 is None else len(paso1),"candidatos":None if candidatos is None else len(candidatos),"h2r":maximo_h2r}}
base=json.dumps(ent,ensure_ascii=False,sort_keys=True).encode(); ent["sha256"]=hashlib.sha256(base).hexdigest()
safe=re.sub(r"[^A-Za-z0-9_-]+","_",CODIGO)
jp=Path(f"TC1_{safe}_resultados.json"); mp=Path(f"TC1_{safe}_reporte.md")
jp.write_text(json.dumps(ent,ensure_ascii=False,indent=2),encoding="utf-8")
lines=["# TC1 — reporte",f"- Estudiante: **{ESTUDIANTE}**",f"- Código: **{CODIGO}**",f"- Puntaje: **{pts}/100**",f"- Nota: **{nota}/5.0**",f"- SHA: `{ent['sha256']}`","","|Control|Estado|Puntos|","|---|---:|---:|"]
for k,v in R.items(): lines.append(f"|{k}|{'OK' if v['ok'] else 'NO'}|{v['puntos']}/{v['maximo']}|")
mp.write_text("\n".join(lines),encoding="utf-8")
print(f"PUNTAJE {pts}/100 · NOTA {nota}/5.0"); print(jp); print(mp)

### Entrega final
Descarga el `.ipynb`, el `.json` y el `.md`. Puedes subirlos a GitHub si quieres, pero **no es requisito**.

**Extensión opcional, sin puntos:** repetir una consulta equivalente en Atlas, Astra o Neo4j Aura. No publiques credenciales.